<a href="https://colab.research.google.com/github/aquastrain/internship_tasks/blob/task-2/multimodal_assistant_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Step 1 - Install dependencies

In [8]:
%%capture
!pip install streamlit pyngrok transformers torch torchvision Pillow scikit-learn numpy

## Step 2 - Write the Streamlit app

In [9]:
%%writefile app.py
# ============================================================
#  Multi-Modal AI Assistant  -  Streamlit app
#  Models: BLIP (captioning), CLIP (similarity), Flan-T5 (LLM)
# ============================================================
import streamlit as st
import torch
import numpy as np
from PIL import Image
import re, textwrap

st.set_page_config(page_title="Multi-Modal AI Assistant",
                   page_icon="robot", layout="wide")

# ── Model loaders (cached) ──────────────────────────────────
@st.cache_resource(show_spinner="Loading BLIP captioner...")
def load_blip():
    from transformers import BlipProcessor, BlipForConditionalGeneration
    proc  = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
    model = BlipForConditionalGeneration.from_pretrained(
                "Salesforce/blip-image-captioning-base")
    model.eval()
    return proc, model

@st.cache_resource(show_spinner="Loading CLIP...")
def load_clip():
    from transformers import CLIPProcessor, CLIPModel
    proc  = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
    model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
    model.eval()
    return proc, model

@st.cache_resource(show_spinner="Loading Flan-T5 LLM...")
def load_llm():
    from transformers import pipeline
    return pipeline("text2text-generation", model="google/flan-t5-base",
                    device=-1, max_new_tokens=300)

# ── Vision helpers ──────────────────────────────────────────
def caption_image(image: Image.Image) -> str:
    # Generate a detailed caption for an image using BLIP
    proc, model = load_blip()
    inputs = proc(image, return_tensors="pt")
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=80, num_beams=5,
                             early_stopping=True)
    return proc.decode(out[0], skip_special_tokens=True)

def clip_score(image: Image.Image, texts: list) -> list:
    # Return cosine similarity scores between image and a list of text labels
    proc, model = load_clip()
    inputs = proc(text=texts, images=image, return_tensors="pt", padding=True)
    with torch.no_grad():
        out = model(**inputs)
    logits = out.logits_per_image[0]          # (num_texts,)
    probs  = logits.softmax(dim=-1).tolist()
    return list(zip(texts, probs))

def describe_image(image: Image.Image) -> dict:
    # Full image analysis: BLIP caption + CLIP scene/object labels
    # Returns a dict with 'caption', 'scene', 'objects'
    caption = caption_image(image)

    scene_labels  = ["indoor scene", "outdoor scene", "nature landscape",
                     "urban environment", "abstract art", "document or text",
                     "food or meal", "people or faces", "animals", "vehicles",
                     "technology or electronics", "sports or activity"]
    scene_scores  = clip_score(image, scene_labels)
    top_scene     = max(scene_scores, key=lambda x: x[1])

    object_labels = ["a person", "a car", "a building", "an animal", "food",
                     "text or document", "a chart or graph", "nature or plants",
                     "water or ocean", "sky or clouds", "technology device", "artwork"]
    obj_scores    = clip_score(image, object_labels)
    top_objects   = sorted(obj_scores, key=lambda x: -x[1])[:3]

    return {
        "caption":  caption,
        "scene":    top_scene,
        "objects":  top_objects,
    }

# ── Reasoning helpers ───────────────────────────────────────
CONFIDENCE_THRESHOLD = 0.35

def build_context_string(history: list, max_turns: int = 6) -> str:
    # Format recent conversation history into a context string
    recent = history[-max_turns:]
    lines  = []
    for turn in recent:
        role = "User" if turn["role"] == "user" else "Assistant"
        lines.append(f"{role}: {turn['content']}")
    return "\n".join(lines)

def is_ambiguous(question: str, image_info: dict) -> tuple:
    # Heuristic ambiguity detection. Returns (is_ambiguous: bool, reason: str)
    q_lower = question.lower().strip()

    # Very short / vague questions
    vague_patterns = [r"^(what|how|why|tell me|explain)\?*$",
                      r"^(this|that|it|here)\?*$"]
    for p in vague_patterns:
        if re.match(p, q_lower):
            return True, "The question is too vague. Please be more specific."

    # Image present but question has no image reference and is generic
    if image_info and len(question.split()) < 4:
        return True, "Could you clarify what aspect of the image you are asking about?"

    return False, ""

def validate_response(response: str, question: str) -> str:
    # Post-generation validation: remove empty/repetitive responses, add caveat if too short
    response = response.strip()
    if not response or response.lower() in [".", "none", "n/a", ""]:
        return "I was unable to generate a confident answer. Could you rephrase your question?"
    if len(response.split()) < 4:
        response += " (Note: I have limited information to give a fuller answer.)"
    # Remove exact repetition of the question inside the answer
    question_clean = question.strip().rstrip("?.")
    if response.lower().startswith(question_clean.lower()):
        response = response[len(question_clean):].strip(": ")
    return response

def generate_response(question: str, image_info: dict,
                      history: list, llm) -> str:
    # Core reasoning pipeline:
    # 1. Check for ambiguity  -> ask clarification
    # 2. Build rich prompt with image context + history
    # 3. Generate with LLM
    # 4. Validate output
    # Step 1 - ambiguity check
    ambig, reason = is_ambiguous(question, image_info)
    if ambig:
        return f"I need a bit more information. {reason}"

    # Step 2 - build prompt
    history_ctx = build_context_string(history)

    if image_info:
        caption  = image_info.get("caption", "")
        scene    = image_info.get("scene", ("unknown", 0))
        objects  = image_info.get("objects", [])
        obj_str  = ", ".join([o[0] for o in objects[:3]])
        img_ctx  = (f"Image description: {caption}. "
                    f"Scene type: {scene[0]} (confidence {scene[1]:.0%}). "
                    f"Prominent elements: {obj_str}.")
    else:
        img_ctx = "No image provided."

    prompt = (
        "You are a helpful, evidence-based multi-modal AI assistant. "
        "You reason carefully over both text and image context before answering. "
        "If you are unsure, say so and explain why.\n\n"
        f"Image context: {img_ctx}\n\n"
        f"Conversation so far:\n{history_ctx}\n\n"
        f"User question: {question}\n\n"
        "Provide a clear, reasoned answer:"
    )

    # Step 3 - generate
    raw = llm(prompt)[0]["generated_text"].strip()

    # Step 4 - validate
    return validate_response(raw, question)

# ── Streamlit UI ────────────────────────────────────────────
st.title("Multi-Modal AI Assistant")
st.caption("Upload an image and chat - the assistant reasons over both vision and text.")

# Session state
if "history"    not in st.session_state: st.session_state.history    = []
if "image_info" not in st.session_state: st.session_state.image_info = None
if "pil_image"  not in st.session_state: st.session_state.pil_image  = None

# Sidebar
with st.sidebar:
    st.header("Settings")
    max_turns = st.slider("Memory turns", 2, 10, 6)
    st.markdown("---")
    st.markdown("**How it works**\n"
                "1. (Optional) Upload an image\n"
                "2. Ask any question\n"
                "3. The assistant reasons over the image + your conversation history\n"
                "4. Ambiguous questions trigger a clarification request\n"
                "5. Responses are validated before display")
    if st.button("Clear conversation"):
        st.session_state.history    = []
        st.session_state.image_info = None
        st.session_state.pil_image  = None
        st.rerun()

# Layout
col1, col2 = st.columns([1, 2])

# ── Left column: image upload + analysis ────────────────────
with col1:
    st.subheader("Image Input")
    uploaded = st.file_uploader("Upload an image (jpg, png, webp)",
                                type=["jpg", "jpeg", "png", "webp"])

    if uploaded:
        pil_img = Image.open(uploaded).convert("RGB")
        st.session_state.pil_image = pil_img
        st.image(pil_img, use_container_width=True)

        if st.button("Analyse image"):
            with st.spinner("Analysing image with BLIP + CLIP..."):
                info = describe_image(pil_img)
                st.session_state.image_info = info

                # Inject image analysis into conversation history
                analysis_msg = (
                    f"[Image analysed] Caption: {info['caption']}. "
                    f"Scene: {info['scene'][0]} ({info['scene'][1]:.0%}). "
                    f"Key elements: {', '.join([o[0] for o in info['objects'][:3]])}."
                )
                st.session_state.history.append({
                    "role": "assistant",
                    "content": analysis_msg
                })

    if st.session_state.image_info:
        info = st.session_state.image_info
        st.markdown("**Image Analysis**")
        st.markdown(f"- **Caption:** {info['caption']}")
        st.markdown(f"- **Scene:** {info['scene'][0]} ({info['scene'][1]:.0%} confidence)")
        st.markdown("**Top Elements (CLIP)**")
        for label, score in info["objects"]:
            st.progress(float(score), text=f"{label}: {score:.0%}")

# ── Right column: chat ───────────────────────────────────────
with col2:
    st.subheader("Conversation")

    # Chat history display
    chat_container = st.container()
    with chat_container:
        for turn in st.session_state.history:
            with st.chat_message(turn["role"]):
                st.markdown(turn["content"])

    # Chat input
    user_input = st.chat_input("Ask about the image or any topic...")
    if user_input:
        st.session_state.history.append({"role": "user", "content": user_input})

        llm = load_llm()
        with st.spinner("Reasoning..."):
            reply = generate_response(
                question   = user_input,
                image_info = st.session_state.image_info,
                history    = st.session_state.history[:-1],  # exclude current user msg
                llm        = llm,
            )

        st.session_state.history.append({"role": "assistant", "content": reply})
        st.rerun()


Overwriting app.py


## Step 3 - Verify the app was written

This just confirms `app.py` exists and has no syntax errors.

In [10]:
import py_compile, os
assert os.path.exists('app.py'), 'app.py not found!'
py_compile.compile('app.py', doraise=True)
print('app.py syntax OK -', os.path.getsize('app.py'), 'bytes')

app.py syntax OK - 10973 bytes


## Step 4 - Launch via pyngrok

Run this cell. It prints a public URL you can open in any browser.

> **First launch** downloads three models (~1.5 GB total). Takes ~3-5 min on Colab.
> Add your ngrok auth token if prompted (free at https://ngrok.com).

In [11]:
import subprocess, time
from pyngrok import ngrok

# ---- Optional: paste your ngrok auth token here ----
# To fix the error, uncomment the line below and replace 'YOUR_TOKEN_HERE' with your actual ngrok auth token.
ngrok.set_auth_token('3FJrMwRr3IeuqGWlYWPNfBkSjoL_51SmmQh82p642pa15of5F')

# Kill any previous Streamlit instance
subprocess.run(['pkill', '-f', 'streamlit'], capture_output=True)
time.sleep(2)

# Launch Streamlit in the background
proc = subprocess.Popen(
    ['streamlit', 'run', 'app.py',
     '--server.port', '8501',
     '--server.headless', 'true',
     '--server.enableCORS', 'false',
     '--server.enableXsrfProtection', 'false'],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)
time.sleep(6)  # wait for Streamlit to start

# Open public tunnel
public_url = ngrok.connect(8501)
print('=' * 60)
print(f'  Assistant is LIVE at: {public_url}')
print('=' * 60)
print('  Keep this cell running. Open the link above in your browser.')

  Assistant is LIVE at: NgrokTunnel: "https://bruising-rift-supernova.ngrok-free.dev" -> "http://localhost:8501"
  Keep this cell running. Open the link above in your browser.
